In [0]:
# Step 1: Set up the tables used in the final review
from pyspark.sql import functions as F
from datetime import datetime, timezone

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

bronze_table = f"{catalog}.{schema_name}.streaming_orders_bronze"
silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"

daily_table = f"{catalog}.{schema_name}.streaming_gold_daily_sales"
product_table = f"{catalog}.{schema_name}.streaming_gold_product_performance"
regional_table = f"{catalog}.{schema_name}.streaming_gold_regional_sales"

audit_table = f"{catalog}.{schema_name}.streaming_data_quality_audit"
test_table = f"{catalog}.{schema_name}.automated_test_history"

report_table = f"{catalog}.{schema_name}.final_validation_report"

validation_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("Validation run:", validation_run_id)
print("Catalog:", catalog)
print("Schema:", schema_name)

In [0]:
# Step 2: Check which project tables are available
required_tables = {
    "Bronze": bronze_table,
    "Silver": silver_table,
    "Gold daily sales": daily_table,
    "Gold product performance": product_table,
    "Gold regional sales": regional_table,
    "Data quality audit": audit_table,
    "Automated test history": test_table
}

validation_results = []

for label, table_name in required_tables.items():
    exists = spark.catalog.tableExists(table_name)

    validation_results.append((
        validation_run_id,
        label,
        "PASS" if exists else "NOT_READY",
        table_name,
        "Table exists" if exists else "Run the notebook that creates this table"
    ))

    print(f"{label}: {'PASS' if exists else 'NOT_READY'}")

In [0]:
# Step 3: Validate the cleaned Silver records
if spark.catalog.tableExists(silver_table):
    silver_df = spark.table(silver_table)

    duplicate_ids = (
        silver_df.groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    missing_ids = silver_df.filter(
        F.col("order_id").isNull()
    ).count()

    invalid_values = silver_df.filter(
        F.col("order_date").isNull()
        | F.col("product_id").isNull()
        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)
        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
    ).count()

    silver_checks = [
        ("Silver unique order IDs", duplicate_ids == 0,
         f"Duplicate IDs: {duplicate_ids}"),
        ("Silver required order IDs", missing_ids == 0,
         f"Missing IDs: {missing_ids}"),
        ("Silver valid order values", invalid_values == 0,
         f"Invalid rows: {invalid_values}")
    ]

    for name, passed, details in silver_checks:
        validation_results.append((
            validation_run_id, name,
            "PASS" if passed else "FAIL",
            silver_table, details
        ))

        print(f"{'PASS' if passed else 'FAIL'}: {name} — {details}")
else:
    print("Silver table is missing. Run Notebook 10 first.")

In [0]:
# Step 4: Compare daily Gold revenue with the Silver source
if (
    spark.catalog.tableExists(silver_table)
    and spark.catalog.tableExists(daily_table)
):
    completed_df = spark.table(silver_table).filter(
        F.lower(F.trim(F.col("status"))) == "completed"
    )

    expected_revenue = completed_df.groupBy("order_date").agg(
        F.round(
            F.sum(F.col("quantity") * F.col("unit_price")), 2
        ).alias("expected_revenue")
    )

    actual_revenue = spark.table(daily_table).select(
        "order_date",
        F.col("total_revenue").alias("actual_revenue")
    )

    mismatches = (
        expected_revenue.alias("e")
        .join(actual_revenue.alias("a"), "order_date", "full")
        .filter(
            F.col("expected_revenue").isNull()
            | F.col("actual_revenue").isNull()
            | (
                F.abs(
                    F.coalesce(F.col("expected_revenue"), F.lit(0.0))
                    - F.coalesce(F.col("actual_revenue"), F.lit(0.0))
                ) > 0.01
            )
        )
        .count()
    )

    passed = mismatches == 0

    validation_results.append((
        validation_run_id,
        "Gold daily revenue reconciliation",
        "PASS" if passed else "FAIL",
        daily_table,
        f"Dates with mismatched revenue: {mismatches}"
    ))

    print("Revenue reconciliation:", "PASS" if passed else "FAIL")
    print("Mismatched dates:", mismatches)
else:
    print("Silver or daily Gold is missing; revenue check was skipped.")

In [0]:
# Step 5: Check whether automated tests have been recorded
if spark.catalog.tableExists(test_table):
    test_history = spark.table(test_table)

    latest_test_run = (
        test_history
        .agg(F.max("test_run_id").alias("latest_run"))
        .first()["latest_run"]
    )

    latest_tests = test_history.filter(
        F.col("test_run_id") == latest_test_run
    )

    failed_count = latest_tests.filter(
        F.col("status") == "FAIL"
    ).count()

    test_status = "PASS" if failed_count == 0 else "FAIL"

    validation_results.append((
        validation_run_id,
        "Automated test history",
        test_status,
        test_table,
        f"Latest run: {latest_test_run}; failed tests: {failed_count}"
    ))

    display(latest_tests.orderBy("status", "test_name"))
else:
    validation_results.append((
        validation_run_id,
        "Automated test history",
        "NOT_READY",
        test_table,
        "Run Notebook 14 first"
    ))

    print("Automated test history is not available yet.")

In [0]:
# Step 6: Combine the checks into one final report
report_df = spark.createDataFrame(
    validation_results,
    [
        "validation_run_id",
        "check_name",
        "status",
        "table_name",
        "details"
    ]
)

display(report_df.orderBy("status", "check_name"))

In [0]:
# Step 7: Keep a history of final validation runs
if not spark.catalog.tableExists(report_table):
    report_df.write.format("delta").saveAsTable(report_table)
else:
    report_df.write.format("delta").mode("append").saveAsTable(report_table)

print("Validation report saved:", report_table)

In [0]:
# Step 8: Summarize the final validation
failed_checks = report_df.filter(
    F.col("status") == "FAIL"
).count()

pending_checks = report_df.filter(
    F.col("status") == "NOT_READY"
).count()

overall_status = (
    "READY FOR REVIEW"
    if failed_checks == 0 and pending_checks == 0
    else "NEEDS ATTENTION"
)

print("Overall status:", overall_status)
print("Failed checks:", failed_checks)
print("Checks not ready:", pending_checks)

display(
    spark.table(report_table)
    .filter(F.col("validation_run_id") == validation_run_id)
    .orderBy("status", "check_name")
)

In [0]:
# Step 18.1: Check which project tables exist
from pyspark.sql import functions as F

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

tables_to_check = [
    "bronze_orders",
    "bronze_products",
    "silver_orders",
    "rejected_orders",
    "gold_daily_sales",
    "gold_product_performance",
    "gold_regional_sales",
    "data_quality_audit",
    "streaming_orders_bronze",
    "streaming_orders_silver",
    "streaming_gold_daily_sales",
    "streaming_gold_product_performance",
    "streaming_gold_regional_sales",
    "streaming_data_quality_audit",
    "streaming_test_history",
    "automated_test_history",
    "pipeline_run_log",
    "final_validation_report"
]

audit_rows = []

for table in tables_to_check:
    full_name = f"{catalog}.{schema_name}.{table}"

    # Check availability before reading a table
    if spark.catalog.tableExists(full_name):
        row_count = spark.table(full_name).count()
        audit_rows.append((table, "EXISTS", row_count))
    else:
        audit_rows.append((table, "MISSING", None))

audit_df = spark.createDataFrame(
    audit_rows,
    ["table_name", "status", "row_count"]
)

display(audit_df.orderBy("status", "table_name"))

In [0]:
# Step 18.2: Review the streaming table schemas
for table in [
    "streaming_orders_bronze",
    "streaming_orders_silver"
]:
    full_name = f"{catalog}.{schema_name}.{table}"

    if spark.catalog.tableExists(full_name):
        print(f"\nSchema for {full_name}")
        spark.table(full_name).printSchema()
    else:
        print(f"\nTable not found: {full_name}")

In [0]:
# Test the shared project configuration
from src.retail_lakehouse.config import get_project_config

config = get_project_config(spark)

print("Catalog:", config["catalog"])
print("Schema:", config["schema"])
print("Streaming Bronze:", config["streaming_bronze"])
print("Streaming Silver:", config["streaming_silver"])